## Spark Configurations

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *

conf = SparkConf()

conf.setAppName("Merge Delta Table") # Name of spark application, useful for logs
conf.set("spark.hadoop.fs.s3a.endpoint","http://minio:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", "admin") # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", "@admin123") # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

## Create Data Frame

In [2]:
data_insert = [("Product A", 300),
               ("Product B", 250),
               ("Product C", 150)
              ]

schema = StructType([
        StructField("product_name", StringType(), True),
        StructField("price", IntegerType(), True)
        ])

data_update = [("Product B", 280),
               ("Product D", 250)
              ]

df_insert = spark.createDataFrame(data=data_insert, schema=schema)
df_update = spark.createDataFrame(data=data_update, schema=schema)

## Data Frame What will insert

In [3]:
df_insert.show()

+------------+-----+
|product_name|price|
+------------+-----+
|   Product A|  300|
|   Product B|  250|
|   Product C|  150|
+------------+-----+



## The Data Frame that will update the insert data

In [4]:
df_update.show()

+------------+-----+
|product_name|price|
+------------+-----+
|   Product B|  280|
|   Product D|  250|
+------------+-----+



## Send to minIO S3 and show

In [5]:
df_insert.write.format("delta").mode("append").save("s3a://bronze/table_merge")
df = spark.read.format("delta").load("s3a://bronze/table_merge").show()

+------------+-----+
|product_name|price|
+------------+-----+
|   Product A|  300|
|   Product C|  150|
|   Product B|  250|
+------------+-----+



## Create Delta Table variable to merge

In [6]:
delta_table = DeltaTable.forPath(spark, "s3a://bronze/table_merge")

## Apply merge

In [7]:
delta_table.alias("target").merge(df_update.alias("source"), "target.product_name = source.product_name")\
            .whenMatchedUpdate(set={"price": "source.price"})\
            .whenNotMatchedInsert(values={"product_name": "source.product_name", "price": "source.price"})\
            .execute()

## Showing the result

In [8]:
df = spark.read.format("delta").load("s3a://bronze/table_merge").show()

+------------+-----+
|product_name|price|
+------------+-----+
|   Product A|  300|
|   Product C|  150|
|   Product B|  280|
|   Product D|  250|
+------------+-----+

